# Phase 4: Simple Research Paper Preprocessing
This notebook follows the Lab1 style: lowercase, tokenize, remove punctuation, remove stopwords, optionally lemmatize, and return clean tokens.

In [1]:
import importlib
import sys
from pathlib import Path

import nltk
import pandas as pd

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing as preprocessing
preprocessing = importlib.reload(preprocessing)

from src.preprocessing import (
    OUTPUT_COLUMNS,
    REQUIRED_COLUMNS,
    build_preprocessed_dataset,
    clean_text,
    preprocess_dataframe,
    preprocess_document,
    remove_stopwords,
    tokenize_text,
    validate_required_columns,
)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DFIT\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [11]:
# Raw text example
raw_text = "BERT, CNN, HPC, GPU, NLP and LLM models are useful!!!"

print("raw text =", raw_text)
print("lowercase =", raw_text.lower())
print("tokens =", tokenize_text(raw_text))
print("without stopwords =", remove_stopwords(tokenize_text(raw_text)))
print("clean text =", clean_text(raw_text))
print("lemmatized text =", clean_text(raw_text, lemmatize=True))


raw text = BERT, CNN, HPC, GPU, NLP and LLM models are useful!!!
lowercase = bert, cnn, hpc, gpu, nlp and llm models are useful!!!
tokens = ['bert', 'cnn', 'hpc', 'gpu', 'nlp', 'and', 'llm', 'models', 'are', 'useful']
without stopwords = ['bert', 'cnn', 'hpc', 'gpu', 'nlp', 'llm', 'models', 'useful']
clean text = bert cnn hpc gpu nlp llm models useful
lemmatized text = bert cnn hpc gpu nlp llm model useful


In [2]:
# Process the research-paper dataset while preserving every required column
DATA_PATH = PROJECT_ROOT / "data" / "processed_data" / "merged_research_papers.csv"
papers = pd.read_csv(DATA_PATH)
validate_required_columns(papers)

processed_papers = preprocess_dataframe(papers, lemmatize=False)

assert all(column in processed_papers.columns for column in REQUIRED_COLUMNS)
assert all(column in processed_papers.columns for column in ["title", "abstract", "clean_text", "category", "year"])
assert len(processed_papers) == len(papers)

print("papers =", len(processed_papers))
print("all required columns present =", True)
processed_papers[["Title", "Abstract", "clean_text", "category", "year"]].head()


papers = 46344
all required columns present = True


,Title,Abstract,clean_text,category,year
0,The role of quantum computing in advancing sci...,NaN,role quantum computing advancing scientific hi...,Quantum Computing Algorithms and Architecture,<NA>
1,Queue wait time prediction in high performance...,Abstract High Performance Computing (HPC) syst...,queue wait time prediction high performance co...,Cloud Computing and Resource Management,<NA>
2,A High-Performance Computing Portal Applied to...,Abstract The continued expansion in size and r...,high-performance computing portal applied 3d e...,Advanced Electron Microscopy Techniques and Ap...,<NA>
3,End-edge-cloud collaborative-driven waste-heat...,NaN,end-edge-cloud collaborative-driven waste-heat...,Cloud Computing and Resource Management,<NA>
4,MorphoCloud: Democratizing Access to High-Perf...,Background: The digitization of biological spe...,morphocloud democratizing access high-performa...,"Genetics, Bioinformatics, and Biomedical Research",<NA>


In [5]:
# Generate the updated local preprocessed dataset
preprocessed_dataset = build_preprocessed_dataset(papers, lemmatize=False)
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed_data" / "preprocessed_research_papers.csv"
preprocessed_dataset.to_csv(OUTPUT_PATH, index=False)

saved_dataset = pd.read_csv(OUTPUT_PATH)
assert len(saved_dataset) == len(papers)
assert all(column in saved_dataset.columns for column in papers.columns)
assert all(column in saved_dataset.columns for column in ["Category", "text"])
assert set(saved_dataset["Top 1% cited"].dropna().unique()).issubset({0, 1})
assert set(saved_dataset["Top 10% cited"].dropna().unique()).issubset({0, 1})
expected_text = (
    saved_dataset.loc[0, "Title"]
    + " "
    + saved_dataset["Abstract"].fillna("").loc[0]
).strip()
assert saved_dataset.loc[0, "text"] == expected_text

print("saved to =", OUTPUT_PATH)
print("rows =", len(saved_dataset))
print("columns =", list(saved_dataset.columns))
print("output top 1% values =", sorted(saved_dataset["Top 1% cited"].unique()))
print("output top 10% values =", sorted(saved_dataset["Top 10% cited"].unique()))
saved_dataset[["Title", "Abstract", "Category", "text"]].head()


saved to = D:\4-1\NLP_Lab\NLP-Based-Research-Paper-Intelligence-System\data\processed_data\preprocessed_research_papers.csv
rows = 46344
columns = ['Title', 'Author', 'Citation count', 'Concept', 'Domain', 'Field', 'Keyword', 'Topic', 'Topic IDs', 'Institution', 'Cited by', 'Cites', 'DOI', 'Abstract', 'Top 1% cited', 'Top 10% cited', 'Category', 'text']
output top 1% values = [np.int64(0), np.int64(1)]
output top 10% values = [np.int64(0), np.int64(1)]


,Title,Abstract,Category,text
0,role quantum computing advancing scientific hi...,NaN,hpc,role quantum computing advancing scientific hi...
1,queue wait time prediction high performance co...,abstract high performance computing hpc system...,hpc,queue wait time prediction high performance co...
2,high-performance computing portal applied 3d e...,abstract continued expansion size resolution v...,hpc,high-performance computing portal applied 3d e...
3,end-edge-cloud collaborative-driven waste-heat...,NaN,hpc,end-edge-cloud collaborative-driven waste-heat...
4,morphocloud democratizing access high-performa...,background digitization biological specimens r...,hpc,morphocloud democratizing access high-performa...
